   
Companion notebook for the blog post. Synthetic discharge summary PDFs - all patient data is fabricated, no real health records are used.

Set the four variables in the config cell before running.

   
## ai_parse_document on Databricks SQL - Companion Notebook
This notebook accompanies the blog post *ai_parse_document on Databricks SQL Looks Like Magic. Here's What to Solve Before Production.*

It demonstrates extracting structured fields from synthetic hospital discharge summaries using `ai_parse_document` and `ai_query`, with pseudonymisation, deduplication, and a Bronze → Silver medallion pattern. The batch cells are illustrative; the Structured Streaming cells show the production approach.

**Requirements:** Databricks Runtime 17.1+, Unity Catalog, a model serving endpoint (e.g. `databricks-claude-sonnet-4`).

In [0]:
# === CONFIGURE BEFORE RUNNING ===
PDF_PATH       = "/Workspace/Users/andy.ho@xebia.com/discharge_pdfs"
CATALOG        = "budgetbricks"
SCHEMA         = "healthcare_demo"
MODEL_ENDPOINT = "databricks-claude-sonnet-4"

In [0]:
# Expose config to SQL cells
dbutils.widgets.text("pdf_path", PDF_PATH)
dbutils.widgets.text("catalog", CATALOG)
dbutils.widgets.text("schema", SCHEMA)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

import hashlib
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Demo only - production should use SHA-256 + secrets manager
def pseudonymise(value: str) -> str:
    if not value:
        return None
    return hashlib.md5(f"DEMO_SALT_{value}".encode()).hexdigest()

spark.udf.register("pseudonymise", pseudonymise, StringType())

<function __main__.pseudonymise(value: str) -> str>

   
### Loading PDFs into Spark

`ai_parse_document` expects a `BINARY` column. How you get there depends on your setup:

* **Option A** - You have a UC Volume or a classic cluster. Run the SQL cell below.
* **Option B** - You're on serverless with workspace files. Skip Option A and run the Python cell instead.

Both produce the same `raw_pdfs` temp view. Run **one**, not both.

In [0]:
%sql
-- Option A: UC Volume or classic cluster

-- CREATE OR REPLACE TEMP VIEW raw_pdfs AS
-- SELECT
--     substring_index(path, '/', -1) AS filename,
--     content
-- FROM read_files(
--     :pdf_path,
--     format => 'binaryFile',
--     pathGlobFilter => '*.pdf'
-- );

In [0]:
# Option B: Serverless workaround

import os
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

rows = [(f, open(os.path.join(PDF_PATH, f), "rb").read())
        for f in sorted(os.listdir(PDF_PATH)) if f.endswith(".pdf")]

raw_pdfs = spark.createDataFrame(rows, StructType([
    StructField("filename", StringType()),
    StructField("content", BinaryType())
]))
raw_pdfs.createOrReplaceTempView("raw_pdfs")

print(f"Loaded {len(rows)} PDFs into raw_pdfs temp view")

Loaded 65 PDFs into raw_pdfs temp view


In [0]:
%sql
    
-- Proof of Concept: ai_parse_document + ai_query in one shot
-- Purely illustrative - returns patient_ref in plain text. Do not store this output.
-- Pseudonymisation happens in the Bronze cells below.

WITH raw AS (
    SELECT
        filename,
        ai_parse_document(content, map('version', '2.0')) AS parsed
    FROM raw_pdfs
    LIMIT 1
),
text_extracted AS (
    SELECT
        filename,
        concat_ws('\n', transform(
            try_cast(parsed:document:elements AS ARRAY<VARIANT>),
            element -> try_cast(element:content AS STRING)
        )) AS full_text
    FROM raw
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
)
SELECT
    filename,
    ai_query(
        'databricks-claude-sonnet-4',
        concat(
            'Extract as JSON: {"patient_ref": "string", "document_date": "YYYY-MM-DD", ',
            '"document_type": "discharge_summary|referral_letter|admission_note", ',
            '"primary_diagnosis": "string", "follow_up_required": true|false}. ',
            'Text may be Dutch, Latin, or English.\n\nDocument:\n', full_text
        ),
        returnType => 'STRING'
    ) AS extracted
FROM text_extracted;

filename,extracted
admission_PT_2024_00502_digital.pdf,"Here is the extracted data in JSON format: ```json { ""patient_ref"": ""PT-2024-00502"", ""document_date"": ""2024-03-28"", ""document_type"": ""admission_note"", ""primary_diagnosis"": ""Subarachnoid haemorrhage"", ""follow_up_required"": true } ``` Note: The `follow_up_required` field is set to `true` based on the fact that the patient is being admitted to the hospital and a management plan is being put in place, which typically involves follow-up care. However, this field is not explicitly stated in the document, so this is an inference based on the context."


In [0]:
# Bronze (batch): Parse PDFs, pseudonymise, deduplicate by content hash
# This batch SQL approach works but requires manual state management.
# The Structured Streaming cells below are the recommended production path.

bronze_df = spark.sql("""
WITH raw AS (
    SELECT
        filename,
        md5(content) AS content_hash,
        ai_parse_document(content, map('version', '2.0')) AS parsed
    FROM raw_pdfs
    ORDER BY rand()
    LIMIT 5
),
text_extracted AS (
    SELECT
        filename,
        content_hash,
        concat_ws('\\n', transform(
            try_cast(parsed:document:elements AS ARRAY<VARIANT>),
            element -> try_cast(element:content AS STRING)
        )) AS full_text
    FROM raw
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
),
ai_extraction AS (
    SELECT
        filename,
        content_hash,
        ai_query(
            'databricks-claude-sonnet-4',
            concat(
                'Extract ONLY the JSON object with these exact fields: ',
                '{{"patient_ref": "patient ID", "document_date": "YYYY-MM-DD", ',
                '"document_type": "discharge_summary|referral_letter|admission_note", ',
                '"primary_diagnosis": "diagnosis", "follow_up_required": true|false}}. ',
                'Return ONLY the JSON, no markdown, no explanation. ',
                'Text may be Dutch, Latin, or English.\\n\\n', full_text
            ),
            returnType => 'STRING'
        ) AS ai_response
    FROM text_extracted
    WHERE full_text IS NOT NULL
),
structured AS (
    SELECT
        filename,
        content_hash,
        try_cast(
            from_json(
                regexp_replace(ai_response, '```(?:json)?\\\\s*|\\\\s*```', ''),
                'struct<patient_ref:string,document_date:string,document_type:string,primary_diagnosis:string,follow_up_required:boolean>'
            ) AS struct<patient_ref:string,document_date:string,document_type:string,primary_diagnosis:string,follow_up_required:boolean>
        ) AS extracted
    FROM ai_extraction
)
SELECT
    filename,
    content_hash,
    'v1' AS prompt_version,
    struct(
        pseudonymise(extracted.patient_ref) AS patient_ref,
        try_to_date(extracted.document_date) AS document_date,
        extracted.document_type AS document_type,
        extracted.primary_diagnosis AS primary_diagnosis,
        extracted.follow_up_required AS follow_up_required
    ) AS extracted,
    current_timestamp() AS extracted_at
FROM structured
""")

bronze_df.createOrReplaceTempView("bronze_documents")

print(f"Bronze: {bronze_df.count()} documents processed")
display(bronze_df)

Bronze: 5 documents processed


filename,content_hash,prompt_version,extracted,extracted_at
nursing_PT_2024_01840_scanned_good.pdf,cf0f34586186403ffec0f0b18cf4e040,v1,"List(464e31584dce54132efa050acd370413, 2024-08-14, nursing_progress_note, right hip hemiarthroplasty, true)",2026-03-08T19:35:30.853Z
discharge_PT_2024_01678_scanned_medium.pdf,1d94a004f808d83434420f77cc5e6bfd,v1,"List(9fd0179819a051113bb4a284773b3fe0, 2024-10-14, discharge_summary, Severe community-acquired pneumonia, right lower lobe, true)",2026-03-08T19:35:30.853Z
discharge_PT_2024_01289_scanned_poor.pdf,0447747aee2cd7b2b97dd56bc7269c58,v1,"List(fb68f538631f55f36c3fb28e26dd1e07, 2024-05-22, discharge_summary, Acute appendicitis, perforated, true)",2026-03-08T19:35:30.853Z
nursing_PT_2024_00987_digital.pdf,efc8ec6dd90c68328d1171ee674d96d7,v1,"List(73aaa3198cd767ea9bc129fe86c76dee, 2024-10-30, admission_note, bronchiolitis, false)",2026-03-08T19:35:30.853Z
referral_PT_2024_02050_scanned_poor.pdf,3f80ab2caded8e11b944a0e167101109,v1,"List(0d5cbba38ccaf77d8c8409f3cf897bd6, 2024-09-22, referral_letter, migraine with aura, true)",2026-03-08T19:35:30.853Z


   
### Bronze: Structured Streaming (Production)

The batch cell above follows the SQL-first instinct - hash each file's bytes, track what's been processed, version by prompt. That works, but it's boilerplate you have to write and get right yourself.

[Structured Streaming](https://docs.databricks.com/en/structured-streaming/index.html) with [Auto Loader](https://docs.databricks.com/en/ingestion/cloud-object-storage/auto-loader/index.html) is a cleaner path. Instead of writing state management by hand, you open a stream from the volume and Spark handles it via checkpoints. New files flow through automatically; nothing gets re-processed.

The pipeline splits into three streaming tasks, each writing to its own Delta table:

1. **Parse** - Auto Loader reads new PDFs, `ai_parse_document` extracts content, `md5(content)` fingerprints raw bytes → `bronze_parsed_raw`
2. **Text extraction** - Filters parse errors, assembles document elements into full text → `bronze_documents_text`
3. **Field extraction** - `ai_query` + JSON parsing + pseudonymisation of `patient_ref` only → `bronze_documents_structured`

Splitting into three tasks pays off when you update the prompt. Parsing is the expensive step - `ai_parse_document` charges per page. When you refine the extraction, reset only the Task 3 checkpoint and rerun `ai_query` on the already-parsed text. The parse cost stays at zero for existing documents.

`trigger(availableNow=True)` processes all pending files then stops, so these cells work in a scheduled notebook or Databricks Workflow.

> Requires a UC Volume path (not workspace files) and a writable checkpoint location.

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, udf, expr, md5, struct
)
from pyspark.sql.types import StringType

checkpoint_path = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
pseudonymise_udf = udf(pseudonymise, StringType())

# Read new files from the volume (only files not yet seen by the checkpoint)
raw_files = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .load(PDF_PATH)
)

# ai_parse_document on each new file, fingerprint raw bytes for dedup
parsed = raw_files.select(
    "path",
    md5("content").alias("content_hash"),
    expr("ai_parse_document(content, map('version', '2.0'))").alias("parsed")
)

# trigger(availableNow=True) processes all pending files then stops
(parsed.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/parse")
    .trigger(availableNow=True)
    .toTable("bronze_parsed_raw")
    .awaitTermination()
)

print("Task 1 complete: bronze_parsed_raw")
display(spark.table("bronze_parsed_raw").select("path", "content_hash").limit(5))

In [0]:
# Filter parse errors, assemble document elements into a single string
parsed_docs = spark.readStream.table("bronze_parsed_raw")

text_extracted = (
    parsed_docs
    .filter(expr("try_cast(parsed:error_status AS STRING) IS NULL"))
    .select(
        "path",
        "content_hash",
        expr("""
            concat_ws('\\n\\n', transform(
                try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                element -> try_cast(element:content AS STRING)
            ))
        """).alias("full_text")
    )
    .filter("full_text IS NOT NULL")
)

(text_extracted.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/text")
    .trigger(availableNow=True)
    .toTable("bronze_documents_text")
    .awaitTermination()
)

print("Task 2 complete: bronze_documents_text")
display(spark.table("bronze_documents_text").select("path", "content_hash", expr("left(full_text, 200) AS text_preview")).limit(5))

In [0]:
# ai_query → strip markdown fences → parse JSON → pseudonymise patient_ref
documents = spark.readStream.table("bronze_documents_text")

# Step 1: Call ai_query
with_response = documents.withColumn(
    "ai_response",
    expr(f"""
        ai_query(
            '{MODEL_ENDPOINT}',
            concat(
                'Extract ONLY the JSON object with these exact fields: ',
                '{{"patient_ref": "patient ID", "document_date": "YYYY-MM-DD", ',
                '"document_type": "discharge_summary|referral_letter|admission_note", ',
                '"primary_diagnosis": "diagnosis", "follow_up_required": true|false}}. ',
                'Return ONLY the JSON, no markdown, no explanation. ',
                'Text may be Dutch, Latin, or English.\\n\\n',
                full_text
            ),
            returnType => 'STRING'
        )
    """)
)

# Step 2: Strip markdown fences, parse JSON into struct
with_parsed = with_response.withColumn(
    "raw_extracted",
    expr("""
        from_json(
            regexp_replace(ai_response, '```(?:json)?\\s*|\\s*```', ''),
            'struct<patient_ref:string,document_date:string,document_type:string,primary_diagnosis:string,follow_up_required:boolean>'
        )
    """)
)

# Step 3: Pseudonymise patient_ref only, cast types, build final struct
final = with_parsed.select(
    "path",
    "content_hash",
    struct(
        pseudonymise_udf(col("raw_extracted.patient_ref")).alias("patient_ref"),
        col("raw_extracted.document_date").cast("date").alias("document_date"),
        col("raw_extracted.document_type").alias("document_type"),
        col("raw_extracted.primary_diagnosis").alias("primary_diagnosis"),
        col("raw_extracted.follow_up_required").alias("follow_up_required")
    ).alias("extracted"),
    lit("v1").alias("prompt_version"),
    current_timestamp().alias("extracted_at")
)

(final.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/query")
    .trigger(availableNow=True)
    .toTable("bronze_documents_structured")
    .awaitTermination()
)

print("Task 3 complete: bronze_documents_structured")
display(spark.table("bronze_documents_structured"))

In [0]:
%sql
CREATE OR REPLACE FUNCTION strip_noise(text STRING)
RETURNS STRING
RETURN regexp_replace(
    regexp_replace(
        text,
        'Vertrouwelijk\\s*-\\s*[\\w\\s]+\\s*-\\s*Pagina \\d+ van \\d+',
        ''
    ),
    '\\n{3,}', '\\n\\n'
);

In [0]:
# Silver: Unnest struct, add business logic, apply cleaning

silver_df = spark.sql("""
SELECT
    extracted.patient_ref AS patient_ref,
    extracted.document_date AS document_date,
    lower(trim(extracted.document_type)) AS document_type,
    strip_noise(extracted.primary_diagnosis) AS primary_diagnosis,
    extracted.follow_up_required AS follow_up_required,
    datediff(current_date(), extracted.document_date) AS document_age_days,
    -- Placeholder: assumes 30-day follow-up window. Extend the prompt to extract
    -- the actual follow-up date from the document if your use case requires it.
    CASE 
        WHEN extracted.follow_up_required = true 
        THEN date_add(extracted.document_date, 30)
        ELSE null 
    END AS follow_up_due_date,
    extracted_at AS processed_at
FROM bronze_documents
WHERE prompt_version = 'v1'
""")

silver_df.createOrReplaceTempView("silver_documents")

print(f"Silver: {silver_df.count()} documents")
display(silver_df)

Silver: 5 documents


patient_ref,document_date,document_type,primary_diagnosis,follow_up_required,document_age_days,follow_up_due_date,processed_at
d0186decc8fa1d107fc156066326d75f,2024-08-05,referral_letter,Chronic lower back pain with L4/5 disc herniation,true,580,2024-09-04,2026-03-08T19:37:51.743Z
d7914208dce92609e7583b483accb560,2024-03-28,admission_note,Subarachnoid haemorrhage,true,710,2024-04-27,2026-03-08T19:37:51.743Z
1cd7de4bc264b3e7821eaf3184a76e7c,2025-02-03,admission_note,New-onset atrial fibrillation with rapid ventricular response,true,398,2025-03-05,2026-03-08T19:37:51.743Z
18c4db58002e92aaff51955e31d38de3,2024-06-29,admission_note,infection,true,617,2024-07-29,2026-03-08T19:37:51.743Z
00ee3ec7cf2a6e2135e4f4d7ea6b7c74,2024-09-22,referral_letter,migraine with aura,true,532,2024-10-22,2026-03-08T19:37:51.743Z


In [0]:
# Monitor null rates to detect LLM extraction failures
# In production, monitor per source hospital - drift at one institution
# hides behind clean data from the others

bronze = spark.table("bronze_documents")
total = bronze.count()

if total == 0:
    print("No documents processed yet")
else:
    null_patient_refs = bronze.filter("extracted.patient_ref IS NULL").count()
    null_rate = null_patient_refs / total
    
    print(f"Total: {total} | Null patient_ref rate: {null_rate:.1%}")
    
    if null_rate > 0.05:
        raise Exception(f"Extraction failure rate {null_rate:.1%} exceeds threshold")
    else:
        print("Quality check passed")

Total: 5 | Null patient_ref rate: 0.0%
Quality check passed
